In [1]:
import pandas as pd
import re
import json
from UNTCSearcher import UNTCSearcher

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

INPUT_PATH = "../treaties/ohchr_instruments_detailed.csv"
OUTPUTPATH = "../treaties/ohchr_instruments_detailed-instit.csv"


In [2]:
web_searcher = UNTCSearcher(headless=True)
OUTPUT_CSV = "untc_title_search_results.csv"


try:
    df_result = web_searcher.search("sugar agreement")
    print(df_result)
finally:
    web_searcher.close()

df_result.to_csv(OUTPUT_CSV, index=False)

 -> 22 rows
        search_term Registration Number  \
0   sugar agreement              I-9369   
1   sugar agreement              I-9369   
2   sugar agreement             I-12951   
3   sugar agreement             I-12951   
4   sugar agreement             A-12951   
5   sugar agreement             A-12951   
6   sugar agreement             A-12951   
7   sugar agreement             A-12951   
8   sugar agreement             A-12951   
9   sugar agreement             I-16200   
10  sugar agreement             I-16200   
11  sugar agreement             A-16200   
12  sugar agreement             I-23225   
13  sugar agreement             I-23225   
14  sugar agreement             I-25811   
15  sugar agreement             I-29467   
16  sugar agreement             I-29467   
17  sugar agreement              A-5534   
18  sugar agreement              A-5534   
19  sugar agreement              A-5534   
20  sugar agreement             A-12951   
21  sugar agreement             A-12951   

In [157]:
# Read dataframe
df = pd.read_csv(INPUT_PATH)

# --------------------------------------------------
# Helper functions
# --------------------------------------------------
def extract_ga_resolution(info):
    if pd.isna(info):
        return None

    # Case 1: already standard slash form, e.g. "A/RES/61/106"
    match = re.search(r'A/RES/\d+/\d+', info, flags=re.IGNORECASE)
    if match:
        return match.group(0).upper()

    # Case 2: "resolution 2391 (XXIII)" or "resolution 260 A (III)"
    # -> number, optional letter (discarded), then (roman numeral)
    match = re.search(
        r'\bresolution\s+(\d+)\s*(?:[A-Z]\s*)?\(([IVXLCDM]+)\)',
        info
    )
    if match:
        number, roman = match.groups()
        return f"A/RES/{number}({roman})"

    # Case 3: "resolution 39/46" (bare slash form, no A/RES/ prefix)
    match = re.search(r'\bresolution\s+(\d+/\d+)', info, flags=re.IGNORECASE)
    if match:
        return f"A/RES/{match.group(1)}"

    return None

def extract_info(text):
    """
    Extract the line immediately following 'BY'.
    """
    if pd.isna(text):
        return None

    match = re.search(
        r"(?im)^BY\s*\n([^\n\r]+)",
        str(text)
    )

    return match.group(1).strip() if match else None


def extract_adoption_date(text):
    """
    Extract the line immediately following 'ADOPTED'.
    """
    if pd.isna(text):
        return None

    match = re.search(
        r"(?im)^ADOPTED\s*\n([^\n\r]+)",
        str(text)
    )

    return match.group(1).strip() if match else None


# --------------------------------------------------
# Create new columns
# --------------------------------------------------

df["info"] = df["content"].apply(extract_info)
# df["adoption_date"] = df["content"].apply(extract_adoption_date) # Not necessary because the 'adoption_by' already captures this
df.rename(columns={"adopted_by": "adoption_date"}, inplace=True)
df["institution"] = None
df["resolution"] = None
df["event"] = None
df["relation"] = None
df["location"] = None

# --------------------------------------------------
# Corrections
# --------------------------------------------------
bangkok = "https://www.ohchr.org/en/instruments-mechanisms/instruments/united-nations-rules-treatment-women-prisoners-and-non-custodial"
df.loc[df["url"] == bangkok, "institution"] = "General Assembly"
df.loc[df["url"] == bangkok, "resolution"] = "A/RES/65/229"
df.loc[df["url"] == bangkok, "event"] = "Sixty-fifth session"

aids = "https://www.ohchr.org/en/instruments-mechanisms/instruments/declaration-commitment-hivaids"
df.loc[df["url"] == aids, "institution"] = "General Assembly"
df.loc[df["url"] == aids, "resolution"] = "A/RES/S-26/2"
df.loc[df["url"] == aids, "event"] = "Twenty-sixth special session"

url = 'https://www.ohchr.org/en/instruments-mechanisms/instruments/standard-rules-equalization-opportunities-persons-disabilities'
df.loc[df["url"] == url, "institution"] = "General Assembly"
df.loc[df["url"] == url, "resolution"] = "A/RES/48/96"
df.loc[df["url"] == url, "event"] = "Forty-eighth session" # Easy to infere becasue of A/RES/48 (session N. 48)

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/slavery-convention"
df.loc[df["url"] == url, "institution"] = "League of Nations"
df.loc[df["url"] == url, "resolution"] = "LoN-1414"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/statute-international-tribunal-prosecution-persons-responsible"
df.loc[df["url"] == url, "institution"] = "Security Council"
df.loc[df["url"] == url, "resolution"] = "S/RES/827(1993)"
df.loc[df["url"] == url, "event"] = "3217th meeting"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/abolition-forced-labour-convention-1957-no-105"
df.loc[df["url"] == url, "institution"] = "International Labour Organisation"
df.loc[df["url"] == url, "event"] = "Fortieth session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-against-discrimination-education"
df.loc[df["url"] == url, "institution"] = "United Nations Educational, Scientific and Cultural Organization"
df.loc[df["url"] == url, "event"] = "General Conference"  # no session number given in info

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/declaration-race-and-racial-prejudice"
df.loc[df["url"] == url, "institution"] = "United Nations Educational, Scientific and Cultural Organization"
df.loc[df["url"] == url, "event"] = "Twentieth session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/discrimination-employment-and-occupation-convention-1958-no-111"
df.loc[df["url"] == url, "institution"] = "International Labour Organisation"
df.loc[df["url"] == url, "event"] = "Forty-second session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/employment-policy-convention-1964-no-122"
df.loc[df["url"] == url, "institution"] = "International Labour Organisation"
df.loc[df["url"] == url, "event"] = "Forty-eighth session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/equal-remuneration-convention-1951-no-100"
df.loc[df["url"] == url, "institution"] = "International Labour Organisation"
df.loc[df["url"] == url, "event"] = "Thirty-fourth session"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/basic-principles-independence-judiciary"
df.loc[df["url"] == url, "event"] = "Seventh United Nations Congress on the Prevention of Crime and the Treatment of Offenders, Milan, 26 August-6 September 1985"
df.loc[df["url"] == url, "location"] = "Milan"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/basic-principles-role-lawyers"
df.loc[df["url"] == url, "event"] = "Eighth United Nations Congress on the Prevention of Crime and the Treatment of Offenders, Havana, Cuba"
df.loc[df["url"] == url, "location"] = "Havana, Cuba"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/basic-principles-use-force-and-firearms-law-enforcement"
df.loc[df["url"] == url, "event"] = "Eighth United Nations Congress on the Prevention of Crime and the Treatment of Offenders, Havana, Cuba, 27 August-7 September 1990"
df.loc[df["url"] == url, "location"] = "Havana, Cuba"

url = "https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-relating-status-stateless-persons"
idx = df.index[df["url"] == url][0]
df.loc[df["url"] == url, "event"] = "Conference of Plenipotentiaries"
df.at[idx, "relation"] = [{
    "A_entity": "E/RES/526(XVII)",
    "relation_type": "convened by",
    "B_entity": idx
}]

forced_labour = "https://www.ohchr.org/en/instruments-mechanisms/instruments/forced-labour-convention-1930-no-29"
df.loc[df["url"] == forced_labour, "institution"] = "International Labour Organisation"
df.loc[df["url"] == forced_labour, "event"] = "Fourteenth session"

freedom_association = "https://www.ohchr.org/en/instruments-mechanisms/instruments/freedom-association-and-protection-right-organize-convention"
df.loc[df["url"] == freedom_association, "institution"] = "International Labour Organisation"
df.loc[df["url"] == freedom_association, "event"] = "Thirty-first session"

indigenous_tribal = "https://www.ohchr.org/en/instruments-mechanisms/instruments/indigenous-and-tribal-peoples-convention-1989-no-169"
df.loc[df["url"] == indigenous_tribal, "institution"] = "International Labour Organisation"
df.loc[df["url"] == indigenous_tribal, "event"] = "Seventy-sixth session"

minimum_age = "https://www.ohchr.org/en/instruments-mechanisms/instruments/minimum-age-convention-1973-no-138"
df.loc[df["url"] == minimum_age, "institution"] = "International Labour Organisation"
df.loc[df["url"] == minimum_age, "event"] = "Fifty-eighth session"

geneva_civilians = "https://www.ohchr.org/en/instruments-mechanisms/instruments/geneva-convention-relative-protection-civilian-persons-time-war"
df.loc[df["url"] == geneva_civilians, "event"] = "Diplomatic Conference for the Establishment of International Conventions for the Protection of Victims of War (Geneva, 21 April - 12 August 1949)"
df.loc[df["url"] == geneva_civilians, "location"] = "Geneva"

geneva_pow = "https://www.ohchr.org/en/instruments-mechanisms/instruments/geneva-convention-relative-treatment-prisoners-war"
df.loc[df["url"] == geneva_pow, "event"] = "Diplomatic Conference for the Establishment of International Conventions for the Protection of Victims of Wa (Geneva, 21 April - 12 August 1949)"
df.loc[df["url"] == geneva_pow, "location"] = "Geneva"

guidelines_prosecutors = "https://www.ohchr.org/en/instruments-mechanisms/instruments/guidelines-role-prosecutors"
df.loc[df["url"] == guidelines_prosecutors, "event"] = "Eighth United Nations Congress on the Prevention of Crime and the Treatment of Offenders"
df.loc[df["url"] == guidelines_prosecutors, "location"] = "Havana, Cuba"

guidelines_children = "https://www.ohchr.org/en/instruments-mechanisms/instruments/guidelines-action-children-criminal-justice-system"
df.loc[df["url"] == guidelines_children, "institution"] = "Economic and Social Council"
df.loc[df["url"] == guidelines_children, "resolution"] = "E/RES/1997/30"

principles_extralegal = "https://www.ohchr.org/en/instruments-mechanisms/instruments/principles-effective-prevention-and-investigation-extra-legal"
df.loc[df["url"] == principles_extralegal, "institution"] = "Economic and Social Council"
df.loc[df["url"] == principles_extralegal, "resolution"] = "E/RES/1989/65"

supp_slavery = "https://www.ohchr.org/en/instruments-mechanisms/instruments/supplementary-convention-abolition-slavery-slave-trade-and"
idx_87 = df.index[df["url"] == supp_slavery][0]
df.loc[df["url"] == supp_slavery, "event"] = "Conference of Plenipotentiaries"
df.at[idx_87, "relation"] = [{
    "A_entity": "E/RES/608(XXI)", # ECOSOC resolution 608 (XXI), 30 April 1956
    "relation_type": "convened by",
    "B_entity": idx_87
}]
df.loc[idx_87, "location"] = "Geneva"

death_penalty_safeguards = "https://www.ohchr.org/en/instruments-mechanisms/instruments/safeguards-guaranteeing-protection-rights-those-facing-death"
df.loc[df["url"] == death_penalty_safeguards, "institution"] = "Economic and Social Council"
df.loc[df["url"] == death_penalty_safeguards, "resolution"] = "E/RES/1984/50"

ictr_statute = "https://www.ohchr.org/en/instruments-mechanisms/instruments/statute-international-criminal-tribunal-prosecution-persons"
df.loc[df["url"] == ictr_statute, "institution"] = "Security Council"
df.loc[df["url"] == ictr_statute, "resolution"] = "S/RES/955(1994)"

collective_bargaining = "https://www.ohchr.org/en/instruments-mechanisms/instruments/right-organise-and-collective-bargaining-convention-1949-no-98"
df.loc[df["url"] == collective_bargaining, "institution"] = "International Labour Organisation"
df.loc[df["url"] == collective_bargaining, "event"] = "Thirty-second session"

worst_forms_child_labour = "https://www.ohchr.org/en/instruments-mechanisms/instruments/worst-forms-child-labour-convention-1999-no-182"
df.loc[df["url"] == worst_forms_child_labour, "institution"] = "International Labour Organisation"
df.loc[df["url"] == worst_forms_child_labour, "event"] = "Eighty-seventh session"

cultural_diversity = "https://www.ohchr.org/en/instruments-mechanisms/instruments/universal-declaration-cultural-diversity"
df.loc[df["url"] == cultural_diversity, "institution"] = "United Nations Educational, Scientific and Cultural Organization"
df.loc[df["url"] == cultural_diversity, "event"] = "Thirty-first session"

human_genome = "https://www.ohchr.org/en/instruments-mechanisms/instruments/universal-declaration-human-genome-and-human-rights"
df.loc[df["url"] == human_genome, "institution"] = "United Nations Educational, Scientific and Cultural Organization"
df.loc[df["url"] == human_genome, "event"] = "Twenty-ninth session"

conciliation_commission = "https://www.ohchr.org/en/instruments-mechanisms/instruments/protocol-instituting-conciliation-and-good-offices-commission-be"
df.loc[df["url"] == conciliation_commission, "institution"] = "United Nations Educational, Scientific and Cultural Organization"

protocol_2014_forced_labour = "https://www.ohchr.org/en/instruments-mechanisms/instruments/protocol-2014-forced-labour-convention-1930"
df.loc[df["url"] == protocol_2014_forced_labour, "institution"] = "International Labour Organization"

protocol_I = "https://www.ohchr.org/en/instruments-mechanisms/instruments/protocol-additional-geneva-conventions-12-august-1949-and"
df.loc[df["url"] == protocol_I, "event"] = (
    "Diplomatic Conference on the Reaffirmation and Development of "
    "International Humanitarian Law applicable in Armed Conflicts"
)
df.loc[df["url"] == protocol_I, "resolution"] = "I-17512"

protocol_II = "https://www.ohchr.org/en/instruments-mechanisms/instruments/protocol-additional-geneva-conventions-12-august-1949-and-0"
df.loc[df["url"] == protocol_II, "event"] = (
    "Diplomatic Conference on the Reaffirmation and Development of "
    "International Humanitarian Law applicable in Armed Conflicts"
)
df.loc[df["url"] == protocol_II, "resolution"] = "I-17513"

rome_statute = "https://www.ohchr.org/en/instruments-mechanisms/instruments/rome-statute-international-criminal-court"
df.loc[df["url"] == rome_statute, "event"] = (
    "United Nations Diplomatic Conference of Plenipotentiaries on the "
    "Establishment of an International Criminal Court"
)

vienna_declaration = "https://www.ohchr.org/en/instruments-mechanisms/instruments/vienna-declaration-and-programme-action"
df.loc[df["url"] == vienna_declaration, "event"] = "World Conference on Human Rights"

reduction_statelessness = "https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-reduction-statelessness"
idx_20 = df.index[df["url"] == reduction_statelessness][0]
df.loc[df["url"] == reduction_statelessness, "event"] = "Conference of Plenipotentiaries"
df.at[idx_20, "relation"] = [{
    "A_entity": "A/RES/896(IX)",        # GA resolution 896 (IX)
    "relation_type": "convened by",
    "B_entity": idx_20
}]

status_refugees = "https://www.ohchr.org/en/instruments-mechanisms/instruments/convention-relating-status-refugees"
idx_22 = df.index[df["url"] == status_refugees][0]
df.loc[df["url"] == status_refugees, "event"] = (
    "United Nations Conference of Plenipotentiaries on the Status of "
    "Refugees and Stateless Persons"
)
df.at[idx_22, "relation"] = [{
    "A_entity": "A/RES/429(V)",         # GA resolution 429 (V), 14 December 1950
    "relation_type": "convened",
    "B_entity": idx_22
}]

eradication_hunger = "https://www.ohchr.org/en/instruments-mechanisms/instruments/universal-declaration-eradication-hunger-and-malnutrition"
idx_96 = df.index[df["url"] == eradication_hunger][0]
df.loc[df["url"] == eradication_hunger, "event"] = "World Food Conference"
df.at[idx_96, "relation"] = [
    {
        "A_entity": "A/RES/3180(XXVIII)",   # GA resolution 3180 (XXVIII), 17 December 1973
        "relation_type": "convened",
        "B_entity": idx_96
    },
    {
        "A_entity": "A/RES/3348(XXIX)",     # GA resolution 3348 (XXIX), 17 December 1974
        "relation_type": "endorsed",
        "B_entity": idx_96
    }
]


# Only touch rows that are still fully untouched by any prior ad hoc correction
mask = df["institution"].isna() & df["event"].isna() & df["resolution"].isna()

extracted = df.loc[mask, "info"].apply(extract_ga_resolution)
df.loc[mask, "resolution"] = extracted

# institution = "General Assembly" only where extraction actually succeeded
df.loc[mask & df["resolution"].notna(), "institution"] = "General Assembly"



In [158]:
print(df.columns)
df.to_csv(OUTPUTPATH, index=False)

Index(['title', 'url', 'adoption_date', 'content', 'pdf_url', 'info',
       'institution', 'resolution', 'event', 'relation', 'location'],
      dtype='str')


In [159]:
result = to_correct[to_correct["info"].str.contains("General Assembly", case=False, na=False)]
print(result[["url", "info"]])

Empty DataFrame
Columns: [url, info]
Index: []


In [160]:
keys = ["convened under", "endorsed by", "pursuance of"]

mask_general_assembly = to_correct["info"].str.contains("General Assembly", case=False, na=False)

# mask for presence of any key phrase
mask_keys = to_correct["info"].str.contains("|".join(keys), case=False, na=False)

result = to_correct[mask_general_assembly & mask_keys]

print(result[["url", "info"]])


Empty DataFrame
Columns: [url, info]
Index: []


In [161]:
# known events
keys = ["convened under", "endorsed by", "pursuance of"]

mask_general_assembly = df["institution"].str.contains("General Assembly", case=False, na=False)

mask_keys = df["institution"].str.contains("|".join(keys), case=False, na=False)

result = df.loc[mask_general_assembly & mask_keys, "institution"].to_list()

result


[]